# Rebuild `psi_ihn_compartments.csv` (input to Figure 5b)

Each PSI-IhN was manually split into an axon and a dendrite compartment, which exist as two
separate segments. Figure 5b needs the root ID of each compartment, so it can count that
compartment's output synapses separately.

The compartments are identified by a **seed coordinate** each, which is stable across
proofreading; the root IDs are not. This notebook reads the segmentation at those
coordinates and writes the current root IDs to `../../data/psi_ihn_compartments.csv`.

## When to re-run this

**Whenever you move to a newer materialization.** In particular:

> Materialization **669** — the newest available when this repository was assembled —
> still had two `islet-cltmr` neurons (seeds `22476_4688_1997` and `24266_4298_1651`)
> merged into a single segment, so their axon and dendrite coordinates resolved to the same
> root ID and their dendritic synapse count double-counted the axon output. **Both have
> since been split manually.** Re-running this notebook against a materialization newer
> than 669 should produce `compartments_split = True` for all 34 neurons, after which
> Figure 5b needs no correction.

Check the summary printed at the end: **all 34 rows should report `True`.**


In [ ]:
import numpy as np
import pandas as pd
import caveclient
import cloudvolume as cv

# Set to None to use the latest materialization, or pin a specific version.
MATERIALIZATION = None
MIP = (32, 32, 45)          # resolution the seed coordinates are given in

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")
if MATERIALIZATION is not None:
    client.version = MATERIALIZATION

print("available materializations:", sorted(client.materialize.get_versions()))
print("using version:", client.version)

seg = cv.CloudVolume(
    "graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr",
    mip=MIP, use_https=True, agglomerate=True, progress=False, fill_missing=True,
)


In [ ]:
OUT_CSV = "../../data/psi_ihn_compartments.csv"

# Seed coordinates are stable across proofreading, so they are the identity here.
existing = pd.read_csv(OUT_CSV)
seeds = existing[["tag", "dend_coord", "axon_coord"]]
print(f"{len(seeds)} PSI-IhNs")


def root_at(coord_str):
    x, y, z = (int(v) for v in coord_str.split("_"))
    return int(np.asarray(seg[x, y, z]).squeeze())


rows = []
for _, r in seeds.iterrows():
    d, a = root_at(r.dend_coord), root_at(r.axon_coord)
    rows.append({"tag": r.tag, "dend_coord": r.dend_coord, "axon_coord": r.axon_coord,
                 "dend_root_id": d, "axon_root_id": a, "compartments_split": d != a})

compartments = pd.DataFrame(rows)
compartments.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}")


In [ ]:
n_bad = int((~compartments.compartments_split).sum())
print(f"neurons with axon and dendrite in separate segments: "
      f"{int(compartments.compartments_split.sum())} / {len(compartments)}")

if n_bad:
    print(f"\n{n_bad} neuron(s) still merged — Figure 5b will need the "
          f"'split only' correction:")
    print(compartments[~compartments.compartments_split][
        ["tag", "dend_coord", "axon_coord", "dend_root_id"]].to_string(index=False))
else:
    print("\nAll compartments are split. Figure 5b needs no correction on this version.")
